# 01 Agent Path Analysis

Purpose: troubleshoot agent flow. For each question this notebook records the
**raw facts** of every LLM call — tool type + `search_query` per call, the
full request bodies, and the final answer — plus the `gate_history` (every
grounding-gate decision: accepted/rejected with confidence and relevance, and
the rejected answer when a gate failed). It then flags abnormal paths where
the actual path differs from the expected one.

Input: the dev-subset QA set (`evaluation/data/qa.jsonl`).

Run cells top to bottom. Each question takes ~30-40s (local LLM via the
proxy).

## 1. Setup

Builds the hybrid search index and the agent from `.env` config, via
`evaluation.notebooks.common`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from evaluation.notebooks.common import build_agent, trace_run, load_qa, sample_qa, setup

setup()
agent, real_call_llm = build_agent()
print("confidence threshold:", agent.confidence_threshold)
print("model:", agent.model)

confidence threshold: 0.55
model: qwen/qwen3.5-9b


## 2. Tracing

`trace_run(agent, query, real_call_llm)` wraps the real `agent.call_llm` for a
single run and returns four things:

- `result` — the final `AgentResult`
- `calls_log` — per call: `items` (each function call the model made — `type`
  (`local` = search_local_knowledge_base, `web` = search_bulbapedia), `name`,
  `search_query`), whether the escalation message was in that call's request,
  and a JSON-safe request-body snapshot
- `escalated` — whether escalation fired at any point in the run
- `gate_history` — every grounding-gate decision in order (`rejected`,
  `confidence`, `relevance`, `rejected_answer`)

`real_call_llm` is the agent's real bound method, captured before any tracing;
each `trace_run` wraps that and restores it afterwards, so no run's calls leak
into another's logs.

## 3. Question set

`nature` classifies what the question *needs* (judgment, kept separate from the
raw trace): **local** (answerable from the local KB), **web** (purely
Bulbapedia), **guardrail** (must be rejected; excluded from the escalation
stats).

`expected` is minimal and unprocessed:
- `hybrid -> answer` — first LLM call returns only the hybrid-search tool call, next call returns the answer
- `hybrid -> web -> answer` — hybrid search, then the model calls web on its own, then answers
- `reject` — must be rejected

In [ ]:
QA = load_qa(PROJECT_ROOT / "evaluation" / "data" / "qa.jsonl")
SAMPLE = 8
QUESTIONS = [
    {"question": q["question"], "nature": "local", "expected": "hybrid -> answer"}
    for q in sample_qa(QA, SAMPLE)
]
QUESTIONS += [
    # --- web-only (Bulbapedia) ---
    {"question": "Who voiced Pikachu in the anime?", "nature": "web", "expected": "hybrid -> web -> answer"},
    {"question": "What is the newest Pokémon introduced in Scarlet/Violet?", "nature": "web", "expected": "hybrid -> web -> answer"},
    # --- guardrail (must reject; excluded from escalation analysis) ---
    {"question": "Who won the 2024 Super Bowl?", "nature": "guardrail", "expected": "reject"},
    {"question": "asdfghjkl", "nature": "guardrail", "expected": "reject"},
    {"question": "Tell me about Abraham Lincoln", "nature": "guardrail", "expected": "reject"},
]

## 4. Run

For each question, print the raw per-call record: call items (type +
`search_query` pairs), `[ESC]` when the escalation message was in that call's
request body, the request body of each call (roles/contents, tool outputs
truncated in this print — the full bodies stay in `trace_results`), the final
answer, and the gate-history block (every gate decision in order, which is
what makes gate-failure diagnosis possible).

In [3]:
trace_results = {}

for i, q in enumerate(QUESTIONS, 1):
    result, calls, escalated, gate_history = trace_run(agent, q["question"], real_call_llm)
    trace_results[i] = {"question": q["question"], "result": result, "calls": calls, "gate_history": gate_history}
    print(f"=== [{i}] {q['nature']} | expected: {q['expected']} ===")
    print(f"Q: {q['question']}")
    for j, c in enumerate(calls, 1):
        items = ", ".join(f"({it['type']}, query={it['search_query']!r})" for it in c["items"]) or "no tool call"
        esc = " [ESC]" if c["escalated"] else ""
        body = " | ".join(
            (m.get("role") or m.get("type") or "?")
            + ":" + (str(m.get("content") or m.get("output") or m.get("arguments") or "")[:70])
            for m in c["request"]
        )
        print(f"  call{j}: items=[{items}]{esc}")
        print(f"    request: {body}")
    status = "rejected" if result.rejected else "accepted"
    print(f"  -> {status}, source={result.source}, confidence={result.confidence and round(result.confidence, 3)}, relevance={result.relevance and round(result.relevance, 3)}")
    print(f"  answer: {(result.answer or '')[:300]}")
    for k, g in enumerate(gate_history, 1):
        gs = "rejected" if g.rejected else "accepted"
        print(f"  gate{k}: {gs} conf={g.confidence and round(g.confidence, 3)} rel={g.relevance and round(g.relevance, 3)} rejected_answer={(g.rejected_answer or '')[:150]!r}")
    print()

=== [1] local | expected: hybrid -> answer ===
Q: What are Bulbasaur's two main types and how does that affect its weaknesses to fire and ice moves?
  call1: items=[(local, query='Bulbasaur types weaknesses')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:What are Bulbasaur's two main types and how does that affect its weakn
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:What are Bulbasaur's two main types and how does that affect its weakn | function_call:{"query":"Bulbasaur types weaknesses"} | function_call_output:[{"name": "Bulbasaur", "text": "Pokémon: Bulbasaur (#1)\nGenus: Seed P
  -> accepted, source=local, confidence=0.619, relevance=0.865
  answer: Bulbasaur's two main types are **Grass** and **Poison**.

This dual typing affects its weaknesses to Fire and Ice moves in the following ways:

*   **Fire Moves:** Bulbasaur is **weak** (takes 2

=== [2] local | expected: hybrid -> answer ===
Q: If I want to catch this Seed Pokémon in the wild, what is its capture rate compared to other species?
  call1: items=[(local, query='Seed Pokémon capture rate')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:If I want to catch this Seed Pokémon in the wild, what is its capture 
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:If I want to catch this Seed Pokémon in the wild, what is its capture  | function_call:{"query":"Seed Pokémon capture rate"} | function_call_output:[{"name": "Bulbasaur", "text": "1.0, fire 2.0, water 0.5, electric 0.5
  -> accepted, source=local, confidence=0.606, relevance=0.654
  answer: Based on the retrieved data, there are a few Pokémon with "Seed" in their genus name (e.g., Bulbasaur is described as having a strange seed planted on its back). However, the specific term "See

=== [3] local | expected: hybrid -> answer ===
Q: Which ability acts as Bulbasaur's hidden ability besides Overgrow, and when would it be most useful?
  call1: items=[(local, query='Bulbasaur abilities hidden ability')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Which ability acts as Bulbasaur's hidden ability besides Overgrow, and
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Which ability acts as Bulbasaur's hidden ability besides Overgrow, and | function_call:{"query":"Bulbasaur abilities hidden ability"} | function_call_output:[{"name": "Bulbasaur", "text": "Pokémon: Bulbasaur (#1)\nGenus: Seed P
  call3: items=[no tool call] [ESC]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Which ability acts as Bulbasaur's hidden ability besides Overgrow, and | function_call:{"query":"Bulbasaur abili

=== [4] local | expected: hybrid -> answer ===
Q: How strong are Grass-type attacks against Bulbasaur given its specific type effectiveness ratios?
  call1: items=[(local, query='Bulbasaur types weaknesses')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:How strong are Grass-type attacks against Bulbasaur given its specific
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:How strong are Grass-type attacks against Bulbasaur given its specific | function_call:{"query":"Bulbasaur types weaknesses"} | function_call_output:[{"name": "Bulbasaur", "text": "Pokémon: Bulbasaur (#1)\nGenus: Seed P
  -> accepted, source=local, confidence=0.714, relevance=0.855
  answer: Based on the retrieved data, Bulbasaur is a dual-type Pokémon with **Grass** and **Poison** typing. The type effectiveness ratios for Grass-type attacks against Bulbasaur are as follows:

*   **A

=== [5] local | expected: hybrid -> answer ===
Q: Does Bulbasaur evolve into a later stage since the data shows it is not classified as a baby Pokémon?
  call1: items=[(local, query='Bulbasaur evolution')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Does Bulbasaur evolve into a later stage since the data shows it is no
  call2: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Does Bulbasaur evolve into a later stage since the data shows it is no | function_call:{"query":"Bulbasaur evolution"} | function_call_output:[{"name": "Bulbasaur", "text": "Pokémon: Bulbasaur (#1)\nGenus: Seed P
  call3: items=[no tool call] [ESC]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Does Bulbasaur evolve into a later stage since the data shows it is no | function_call:{"query":"Bulbasaur evolution"} | function_call_output

...

=== [21] web | expected: hybrid -> web -> answer ===
Q: Who voiced Pikachu in the anime?
  call1: items=[(local, query='Pikachu voice actor anime')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Who voiced Pikachu in the anime?
  call2: items=[(web, query='Pikachu voice actor anime')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Who voiced Pikachu in the anime? | function_call:{"query":"Pikachu voice actor anime"} | function_call_output:[{"name": "Pikachu-Gmax", "text": "Pokémon: Pikachu-Gmax (#10199)\nGen
  call3: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Who voiced Pikachu in the anime? | function_call:{"query":"Pikachu voice actor anime"} | function_call_output:[{"name": "Pikachu-Gmax", "text": "Pokémon: Pikachu-Gmax (#10199)\nGen | function_call:{"query":"Pikachu voice actor anime"} | function_

=== [22] web | expected: hybrid -> web -> answer ===
Q: What is the newest Pokémon introduced in Scarlet/Violet?
  call1: items=[(local, query='newest Pokémon Scarlet Violet introduction')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:What is the newest Pokémon introduced in Scarlet/Violet?
  call2: items=[(web, query='newest Pokémon Scarlet Violet Paldea region introduction')]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:What is the newest Pokémon introduced in Scarlet/Violet? | function_call:{"query":"newest Pokémon Scarlet Violet introduction"} | function_call_output:[{"name": "Minior-Violet", "text": "Pokémon: Minior-Violet (#10142)\nG
  call3: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:What is the newest Pokémon introduced in Scarlet/Violet? | function_call:{"query":"newest Pokémon Scarlet Vio

=== [23] guardrail | expected: reject ===
Q: Who won the 2024 Super Bowl?
  call1: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Who won the 2024 Super Bowl?
  -> rejected, source=None, confidence=None, relevance=None
  answer: I'm a Pokémon knowledge assistant — I can answer questions about Pokémon stats, types, weaknesses, abilities, evolutions, and type matchups. I can't predict battle outcomes, access save files, help with cheating, or answer non-Pokémon topics. Try asking about a specific Pokémon!
  gate1: rejected conf=None rel=None rejected_answer=''



=== [24] guardrail | expected: reject ===
Q: asdfghjkl
  call1: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:asdfghjkl
  call2: items=[no tool call] [ESC]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:asdfghjkl | user:Your previous answer was not grounded in the retrieved documents. Call
  -> rejected, source=None, confidence=0.0, relevance=0.735
  answer: I'm a Pokémon knowledge assistant — I can answer questions about Pokémon stats, types, weaknesses, abilities, evolutions, and type matchups. I can't predict battle outcomes, access save files, help with cheating, or answer non-Pokémon topics. Try asking about a specific Pokémon!
  gate1: rejected conf=0.0 rel=0.031 rejected_answer='I\'m not sure what you\'re trying to ask. Could you please provide a question related to Pokémon? For example: "What are Pikachu\'s stats?" or "How does '
  gate2: rejected co

=== [25] guardrail | expected: reject ===
Q: Tell me about Abraham Lincoln
  call1: items=[no tool call]
    request: developer:You are a Pokémon knowledge assistant that answers questions about the | user:Tell me about Abraham Lincoln
  -> rejected, source=None, confidence=None, relevance=None
  answer: I'm a Pokémon knowledge assistant — I can answer questions about Pokémon stats, types, weaknesses, abilities, evolutions, and type matchups. I can't predict battle outcomes, access save files, help with cheating, or answer non-Pokémon topics. Try asking about a specific Pokémon!
  gate1: rejected conf=None rel=None rejected_answer=''



## 5. Summary (derived — raw data is in section 4)

Escalation rate and per-nature outcomes. Guardrail questions are excluded from
the escalation stats.

In [4]:
import pandas as pd

rows = []
for i, q in enumerate(QUESTIONS, 1):
    t = trace_results[i]
    r = t["result"]
    n_web = sum(
        1 for c in t["calls"] for it in c["items"] if it["type"] == "web"
    )
    rows.append({
        "#": i,
        "question": q["question"],
        "nature": q["nature"],
        "expected": q["expected"],
        "llm calls": len(t["calls"]),
        "escalated": any(c["escalated"] for c in t["calls"]),
        "web calls (any)": n_web,
        "rejected": r.rejected,
        "source": r.source,
        "confidence": round(r.confidence, 3) if r.confidence else None,
        "relevance": round(r.relevance, 3) if r.relevance is not None else None,
    })

df = pd.DataFrame(rows)
df

,#,question,nature,expected,llm calls,escalated,web calls (any),rejected,source,confidence,relevance
0,1,What are Bulbasaur's two main types and how do...,local,hybrid -> answer,2,False,0,False,local,0.619,0.865
1,2,If I want to catch this Seed Pokémon in the wi...,local,hybrid -> answer,2,False,0,False,local,0.606,0.654
2,3,Which ability acts as Bulbasaur's hidden abili...,local,hybrid -> answer,3,True,0,False,local,0.593,0.733
3,4,How strong are Grass-type attacks against Bulb...,local,hybrid -> answer,2,False,0,False,local,0.714,0.855
4,5,Does Bulbasaur evolve into a later stage since...,local,hybrid -> answer,3,True,0,True,NaN,0.522,0.889
5,6,"Ivysaur is a Grass and Poison type, so does it...",local,hybrid -> answer,3,True,0,True,NaN,0.432,0.936
6,7,"Since Ivysaur has the Overgrow ability, what h...",local,hybrid -> answer,4,True,1,False,local+web,0.721,0.778
7,8,"If I want to evolve Ivysaur into Venusaur, do ...",local,hybrid -> answer,4,True,1,False,local+web,0.680,0.897
8,9,Ivysaur is a quadruped shape with green colori...,local,hybrid -> answer,2,False,0,False,local,0.566,0.835
9,10,With a base stat total of 405 and Chlorophyll ...,local,hybrid -> answer,2,False,0,False,local,0.568,0.838


## 6. Abnormal detection

Flags rows where the actual path differs from the expected one, and the
derived summary (in-scope questions, escalation count/%, web-without-
escalation, per-nature outcomes, guardrail outcomes).

In [5]:
def expected_ok(row):
    if row["expected"] == "reject":
        return bool(row["rejected"])
    if row["expected"] == "hybrid -> web -> answer":
        return (not row["escalated"]) and row["web calls (any)"] > 0 and not row["rejected"]
    return (not row["escalated"]) and row["web calls (any)"] == 0 and not row["rejected"]


def abnormal_reasons(row):
    reasons = []
    if row["expected"] == "reject":
        if not row["rejected"]:
            reasons.append("accepted (expected reject)")
    else:
        if row["escalated"]:
            reasons.append("escalation fired")
        if row["rejected"]:
            reasons.append("rejected (answer failed grounding)")
        if row["web calls (any)"] > 0 and "web" not in row["expected"]:
            reasons.append("web called (expected local only)")
        if row["web calls (any)"] == 0 and "web" in row["expected"]:
            reasons.append("web not called (expected model-initiated web)")
    return "; ".join(reasons) or "ok"


abnormal = df[~df.apply(expected_ok, axis=1)].copy()
abnormal["reasons"] = abnormal.apply(abnormal_reasons, axis=1)
print(f"abnormal: {len(abnormal)} of {len(df)} — actual path differs from expected\n")
for _, r in abnormal.iterrows():
    print(f"[{r['#']:2d}] {r['nature']:13s} | expected {r['expected']:22s} | calls={r['llm calls']} "
          f"| esc={str(r['escalated']):5s} web={r['web calls (any)']} rejected={str(r['rejected']):5s} "
          f"| source={r['source']} | conf={r['confidence']} rel={r['relevance']}")
    print(f"     Q: {r['question']}")
    print(f"     why: {r['reasons']}")

main = df[df["nature"] != "guardrail"]
n = len(main)
n_esc = main["escalated"].sum()
n_web_self = ((main["web calls (any)"] > 0) & (~main["escalated"])).sum()
print(f"in-scope questions: {n}")
print(f"escalated: {n_esc} ({n_esc / n:.0%})")
print(f"web searched WITHOUT escalation (model-initiated): {n_web_self}")
print()
print("per-nature outcomes:")
print(main.groupby("nature")["rejected"].agg(["count", "sum"]).rename(columns={"count": "n", "sum": "rejected"}))
print()
print("guardrail (must reject):")
print(df[df["nature"] == "guardrail"][["#", "rejected", "source"]].to_string(index=False))

abnormal: 12 of 25 — actual path differs from expected

[ 3] local         | expected hybrid -> answer       | calls=3 | esc=True  web=0 rejected=False | source=local | conf=0.593 rel=0.733
     Q: Which ability acts as Bulbasaur's hidden ability besides Overgrow, and when would it be most useful?
     why: escalation fired
[ 5] local         | expected hybrid -> answer       | calls=3 | esc=True  web=0 rejected=True  | source=nan | conf=0.522 rel=0.889
     Q: Does Bulbasaur evolve into a later stage since the data shows it is not classified as a baby Pokémon?
     why: escalation fired; rejected (answer failed grounding)
[ 6] local         | expected hybrid -> answer       | calls=3 | esc=True  web=0 rejected=True  | source=nan | conf=0.432 rel=0.936
     Q: Ivysaur is a Grass and Poison type, so does it take double damage from Ice moves?
     why: escalation fired; rejected (answer failed grounding)
[ 7] local         | expected hybrid -> answer       | calls=4 | esc=True  web=1 rej

## 7. Gate-history deep-dive

For the abnormal rows only, dump the full `gate_history` in order so a gate
failure can be diagnosed: the rejected flag, confidence, relevance, the
rejected answer that failed grounding, and the accepted answer when the row
ended accepted.

In [6]:
for _, r in abnormal.iterrows():
    i = r["#"]
    print(f"=== [{i}] {r['question']}")
    for k, g in enumerate(trace_results[i]["gate_history"], 1):
        gs = "rejected" if g.rejected else "accepted"
        print(f"  gate{k}: {gs} conf={g.confidence and round(g.confidence, 3)} rel={g.relevance and round(g.relevance, 3)}")
        print(f"    rejected_answer: {(g.rejected_answer or '')[:150]!r}")
    if not r["rejected"]:
        print(f"    accepted answer: {(trace_results[i]['result'].answer or '')[:150]!r}")
    print()

in_scope = df[df["nature"] != "guardrail"]
n = len(in_scope)
n_rej = in_scope["rejected"].sum()
n_esc = in_scope["escalated"].sum()
print(f"in-scope rejection rate: {n_rej} / {n} = {n_rej / n:.0%}")
print(f"escalation rate: {n_esc} / {n} = {n_esc / n:.0%}")

=== [3] Which ability acts as Bulbasaur's hidden ability besides Overgrow, and when would it be most useful?
  gate1: rejected conf=0.502 rel=0.763
    rejected_answer: "Bulbasaur's hidden ability is **Chlorophyll**.\n\nThis ability acts as a hidden ability because it is not available on Bulbasaur in its standard form (o"
  gate2: accepted conf=0.593 rel=0.733
    rejected_answer: ''
    accepted answer: "Based on the local knowledge base, Bulbasaur's hidden ability is **Chlorophyll**.\n\nThis ability would be most useful in **sunlight** conditions (such "

=== [5] Does Bulbasaur evolve into a later stage since the data shows it is not classified as a baby Pokémon?
  gate1: rejected conf=0.501 rel=0.902
    rejected_answer: 'Yes, Bulbasaur does evolve into later stages. According to the data, its evolution chain is: **Bulbasaur → Ivysaur → Venusaur**. \n\nThe fact that it is'
  gate2: rejected conf=0.522 rel=0.889
    rejected_answer: 'Based on the retrieved data, Bulbasaur does evolve